# 06 — Generate returns

**Business objective:** a rising return rate is a classic secondary signal
alongside a revenue drop (unhappy customers both buy less and return more).
We engineer a mild uptick in Sydney's return rate during the anomaly window
to make the dataset internally consistent.

**What we're generating:** returns sampled from a subset of completed orders,
at a base ~5% rate, rising to ~8% for Sydney orders placed during the anomaly
window.

In [1]:
import sys
sys.path.insert(0, '..')
from notebooks import nb_config as cfg

import pandas as pd
import numpy as np

rng = np.random.default_rng(cfg.SEED + 5)

orders = pd.read_csv(cfg.DATA_DIR / "orders.csv", parse_dates=["order_date"])
stores = pd.read_csv(cfg.DATA_DIR / "stores.csv")
products = pd.read_csv(cfg.DATA_DIR / "products.csv")

orders = orders.merge(stores[["store_id", "city"]], on="store_id")
end_date = orders["order_date"].max()
anomaly_start = end_date - pd.DateOffset(months=cfg.ANOMALY_MONTHS)
completed = orders[orders["order_status"] == "completed"].copy()

## Generation logic

In [2]:
def return_prob(row):
    if row["city"] == cfg.ANOMALY_CITY and row["order_date"] >= anomaly_start:
        return 0.08
    return 0.05

completed["return_prob"] = completed.apply(return_prob, axis=1)
is_returned = rng.random(len(completed)) < completed["return_prob"].values
returned_orders = completed[is_returned]

reasons = ["Wrong size", "Changed mind", "Damaged item", "Not as described", "Late delivery"]

returns = pd.DataFrame({
    "return_id": range(1, len(returned_orders) + 1),
    "order_id": returned_orders["order_id"].values,
    "product_id": rng.choice(products["product_id"], size=len(returned_orders)),
    "return_date": returned_orders["order_date"].values + pd.to_timedelta(rng.integers(1, 14, size=len(returned_orders)), unit="D"),
    "return_reason": rng.choice(reasons, size=len(returned_orders)),
})
print(f"Generated {len(returns):,} returns out of {len(completed):,} completed orders")
returns.head()

Generated 1,707 returns out of 33,845 completed orders


,return_id,order_id,product_id,return_date,return_reason
0,1,43,93,2025-01-14,Not as described
1,2,46,54,2025-01-03,Changed mind
2,3,74,97,2025-01-07,Late delivery
3,4,81,190,2025-01-16,Late delivery
4,5,82,78,2025-01-08,Not as described


## Sanity check: Sydney return rate vs others in the anomaly window

In [3]:
check = completed[completed["order_date"] >= anomaly_start].copy()
check["returned"] = is_returned[completed["order_date"] >= anomaly_start]
check.groupby("city")["returned"].mean().sort_values(ascending=False)

city
Sydney       0.075294
Perth        0.054422
Brisbane     0.053837
Melbourne    0.050175
Adelaide     0.049826
Canberra     0.049451
Name: returned, dtype: float64

## Validation

In [4]:
assert returns["return_id"].is_unique
assert returns["order_id"].isin(orders["order_id"]).all()
assert returns["product_id"].isin(products["product_id"]).all()
assert returns.isnull().sum().sum() == 0
print("All validation checks passed.")

All validation checks passed.


In [5]:
out_path = cfg.DATA_DIR / "returns.csv"
returns.to_csv(out_path, index=False)
print(f"Wrote {len(returns):,} rows to {out_path}")

Wrote 1,707 rows to /home/claude/project/eaida/data/raw/returns.csv


## Summary

Generated returns at a 5% base rate, rising to 8% for Sydney orders in the
anomaly window — confirmed in the sanity check above. This gives the dataset
a second, correlated signal alongside the order-volume drop. Saved to
`data/raw/returns.csv`.